In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.11 Random Numbers and Monte Carlo Integration

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.11",
    title="Random Numbers and Monte Carlo Integration",
    blurb="A deterministic machine manufactures randomness, and randomness "
    "computes integrals. We build a generator from one multiplication, watch "
    "a famous one fail in fifteen flat planes, and meet the 1/√N law that "
    "makes chance the only method left standing in high dimensions.",
    difficulty="intermediate",
    estimate="100–130 min",
)

## Notebook overview

Every stochastic computation in this course, from the statistical mechanics
of Volume V to the path-integral Monte Carlo of Volume VII, rests on two
ideas this notebook builds from nothing: that a deterministic computer can
manufacture numbers which *behave* random, and that averages over such
numbers *converge* on integrals. Neither is obvious. The first is a small
miracle of number theory with famous failures (we will reproduce the most
famous one, in three dimensions, where it falls into fifteen flat planes);
the second is the law of large numbers wearing its computational face, with
an error that shrinks as $1/\sqrt N$ no matter how many dimensions the
integral lives in. That dimension-independence is the whole point: the
quadrature rules of [§0.3](quadrature-differentiation.ipynb) are unbeatable
on a line and hopeless in thirty dimensions, and Monte Carlo is how physics
computes where grids cannot go.

We build a linear congruential generator from one line of integer
arithmetic and certify it against its designers' own published test value;
break its most notorious cousin; test randomness like a skeptic (moments,
bins, correlations); transform uniform deviates into exponential, Gaussian,
and semicircular ones; measure the $1/\sqrt N$ law and set it against the
trapezoid rule; estimate hypersphere volumes up to ten dimensions; and end
with importance sampling, the doorway through which Volume V's Metropolis
algorithm will walk. The standard references are Numerical Recipes ch. 7
{cite}`numrecipes` and Park & Miller's classic paper {cite}`parkmiller1988`.

A note on reading the checks in this notebook: a validation compares a
result to an expected fact. A ✗ does not by itself mean the answer is
wrong; it means the output did not match what the check expected, which may
be a genuine error, a different-but-valid convention, or a statistical
fluctuation pushed past its tolerance. Treat a ✗ as a prompt to locate the
discrepancy. Passing is strong evidence, not proof. Statistical checks
here use wide (4σ) windows with fixed seeds, so they are deterministic in
this notebook as shipped.

## Theory in brief

**Pseudorandom numbers.** A *linear congruential generator* (LCG) iterates
integer arithmetic,

```{math}
:label: eq-mc-lcg
x_{k+1} \;=\; (a\,x_k + c) \bmod m ,
```

and returns $u_k = x_k/m \in [0, 1)$. The sequence is perfectly
deterministic (that is what makes results reproducible) and eventually
periodic; the art is choosing $a$, $c$, $m$ so the period is long and the
numbers pass every statistical test one throws at them. The *minimal
standard* generator of Park & Miller {cite}`parkmiller1988` uses $a =
16807$, $c = 0$, $m = 2^{31} - 1$ and survives decades of scrutiny; IBM's
RANDU ($a = 65539$, $c = 0$, $m = 2^{31}$) is the canonical disaster:
consecutive triples $(u_k, u_{k+1}, u_{k+2})$ obey the *exact* integer
identity

```{math}
:label: eq-mc-randu
x_{k+2} \;=\; 6\,x_{k+1} - 9\,x_k \pmod{2^{31}} ,
```

so every triple lies on one of just fifteen parallel planes in the unit
cube {cite}`marsaglia1968`. Modern generators (NumPy's default PCG64,
reached through `numpy.random.default_rng`) have periods near $2^{128}$
and no such structure; the course's standing rule of seeding every
generator is precisely the determinism of {eq}`eq-mc-lcg` used *for* us.

**Transforming distributions.** With uniform $U$ in hand, other densities
follow. The *inverse transform*: if $F$ is a cumulative distribution
function, $X = F^{-1}(U)$ has density $F'$; for the exponential
distribution $p(x) = \lambda e^{-\lambda x}$ this reads

```{math}
:label: eq-mc-inverse
X \;=\; -\ln(1 - U)/\lambda .
```

The *Box–Muller transform* manufactures exact Gaussians from two uniforms,

```{math}
:label: eq-mc-boxmuller
Z_1 = \sqrt{-2\ln U_1}\,\cos(2\pi U_2),
\qquad
Z_2 = \sqrt{-2\ln U_1}\,\sin(2\pi U_2),
```

both $\sim\mathcal N(0,1)$ and independent. And *rejection sampling* draws
from any bounded density $p$ on $[a, b]$ with no inverse at all: propose
$(x, y)$ uniformly in the enclosing rectangle, keep $x$ when $y < p(x)$.

**Monte Carlo integration.** The mean-value estimator turns an integral
into an average,

```{math}
:label: eq-mc-estimator
\int_a^b f(x)\,dx \;\approx\; (b-a)\,\frac{1}{N}\sum_{k=1}^{N} f(x_k),
\qquad x_k \sim \mathrm{Uniform}(a,b),
```

with standard error $(b-a)\,\sigma_f/\sqrt N$, where $\sigma_f^2$ is the
variance of $f$ under the sampling density. The $1/\sqrt N$ is the central
limit theorem (Volume V derives it as physics in
[§5.3](../05-classical-stat-mech/large-n-limit.ipynb)); its power is what it
*lacks*: any reference to dimension. A product trapezoid grid at $n$ points
per axis costs $n^d$ evaluations for error $\mathcal O(n^{-2})$, i.e. error
$\mathcal O(N^{-2/d})$ at total cost $N$; Monte Carlo's $N^{-1/2}$ wins for
every $d > 4$, and in the $10^{23}$-dimensional integrals of statistical
mechanics it is the only method there is.

**Importance sampling.** Sampling from a density $w$ shaped like the
integrand slashes the variance:

```{math}
:label: eq-mc-importance
\int f(x)\,dx \;=\; \int \frac{f(x)}{w(x)}\,w(x)\,dx
\;\approx\; \frac{1}{N}\sum_{k=1}^{N} \frac{f(x_k)}{w(x_k)},
\qquad x_k \sim w ,
```

exact for any $w > 0$ where $f \neq 0$, and dramatically better when $f/w$
is nearly constant. Its limit defines the frontier this notebook stops at:
when the ideal weight is a density one cannot sample directly (a Boltzmann
factor with an unknown normalization), a cleverer machine is needed. That
machine is the Metropolis algorithm, and Volume V builds it where its
physics lives ([§5.8](../05-classical-stat-mech/partition-function.ipynb)).

## Setup

All randomness is seeded (`numpy.random.default_rng` with named seeds per
study), so every number and every histogram in this notebook reproduces
exactly. The LCGs are implemented in exact integer arithmetic; floats
appear only when a state is divided by the modulus.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from scipy import stats
from scipy.integrate import quad
from scipy.special import gamma as gamma_fn

from ecp import animate, draw, validate

rng = np.random.default_rng(0)  # master generator for the darts opener

# Park & Miller "minimal standard" and IBM RANDU parameters (Theory in brief)
A_PM, M_PM = 16807, 2**31 - 1
A_RANDU, M_RANDU = 65539, 2**31


def lcg_sequence(a, c, m, seed, n):
    """Generate n states of the linear congruential generator (a, c, m).

    Iterates x_{k+1} = (a x_k + c) mod m in exact Python integer arithmetic
    (no overflow, no rounding): the deterministic heart of eq-mc-lcg. The
    floats handed to statistics come later, as x/m.

    Parameters
    ----------
    a, c, m : int
        Multiplier, increment, and modulus of the generator.
    seed : int
        Starting state x_0 (not returned).
    n : int
        Number of states to generate.

    Returns
    -------
    numpy.ndarray
        The states x_1 … x_n as int64 (all < m ≤ 2^31, safely in range).
    """
    out = np.empty(n, dtype=np.int64)
    x = seed
    for k in range(n):
        x = (a * x + c) % m
        out[k] = x
    return out


def mc_mean_value(f, a, b, n, generator):
    """Mean-value Monte Carlo estimate of ∫_a^b f(x) dx, with its error bar.

    Implements eq-mc-estimator: draw n uniform points, average f, scale by
    (b − a); the standard error is (b − a)·std(f)/√n, estimated from the
    same sample. The estimator every stochastic method in this course
    refines.

    Parameters
    ----------
    f : callable
        Integrand, vectorized over a numpy array.
    a, b : float
        Integration limits.
    n : int
        Number of sample points.
    generator : numpy.random.Generator
        Seeded generator supplying the uniforms.

    Returns
    -------
    tuple of float
        ``(estimate, standard_error)``.
    """
    x = generator.uniform(a, b, size=n)
    fx = f(x)
    est = (b - a) * float(np.mean(fx))
    sem = (b - a) * float(np.std(fx, ddof=1)) / np.sqrt(n)
    return est, sem

## Exercise 1 — π from darts

The oldest Monte Carlo computation makes the estimator {eq}`eq-mc-estimator`
visible before any formalism. Throw $N$ points uniformly into the unit
square $[0,1]^2$; the fraction landing inside the quarter disk
$x^2 + y^2 \le 1$ estimates its area $\pi/4$, because a uniform point's
probability of landing in a region *is* that region's area. The count of
hits is binomial, so the estimator $\hat p$ carries the standard error
$\sqrt{\hat p(1-\hat p)/N}$, and $4\hat p$ estimates $\pi$ with four times
that. {numref}`fig-mc-darts` shows the geometry with the first 400 throws.

**Part a)** With `rng.uniform(0.0, 1.0, size=(N, 2))` and $N = 10^6$, count
hits via the boolean mean of $x^2 + y^2 \le 1$ (`numpy.mean` on the
comparison), form $\hat\pi = 4\hat p$ and its standard error
$4\sqrt{\hat p(1-\hat p)/N}$, and verify $|\hat\pi - \pi|$ lies within
$4$ standard errors (a deterministic check at this seed; the window says
what *any* seed should satisfy 99.99% of the time).

**Part b)** Verify the scaling claim coarsely before Exercise 5 measures it
properly: with the same binomial formula $4\sqrt{\hat p(1-\hat p)/N}$ at
both sizes, the standard error at $N = 10^6$, about $1.6\times10^{-3}$,
is ten times smaller than at $N = 10^4$: 100× the work for 10× the
precision. That exchange rate is the price of the method, and Exercise 5's
fitted slope makes it exact.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(pi_hat - np.pi) < 4.0 * pi_sem,
    "a million darts estimate π within four binomial standard errors",
    f"π̂ = {pi_hat:.6f} ± {pi_sem:.6f}, pull {(pi_hat - np.pi) / pi_sem:+.2f}σ",
)
validate.close(
    sem_small / pi_sem,
    10.0,
    "the exchange rate: 100× the samples buy 10× the precision (the 1/√N law, "
    "measured coarsely)",
    rtol=5e-2,
)

## Exercise 2 — A generator from one multiplication, and a famous corpse

Where do the "uniform" numbers come from? Equation {eq}`eq-mc-lcg` is the
whole machine, and this exercise builds two instances of it in exact
integer arithmetic: one good, one notorious. The claim that a
deterministic sequence can *behave* random is audited in Exercise 3; here
we certify the machinery itself, the way its designers did.

**Part a)** Implement the Park–Miller minimal standard ($a = 16807$,
$c = 0$, $m = 2^{31} - 1$) with `lcg_sequence` from seed $x_0 = 1$. Verify
the first three states are exactly $16807$, $282475249$, $1622650073$, and
the ten-thousandth state is exactly $1043618065$: the certification value
Park & Miller published for this purpose {cite}`parkmiller1988`. Integer
arithmetic either reproduces it bit-for-bit or the implementation is wrong;
there is no tolerance.

**Part b)** Periods are finite. For the toy generator $a = 5$, $c = 1$,
$m = 32$ from seed $7$, generate 64 states with `lcg_sequence` and locate
the period as the first index where the state returns to its first value
(`numpy.flatnonzero` on the equality), verifying it is exactly $32$, the maximum possible (visiting *every* residue
once, per the Hull–Dobell theorem, since $c$ is odd and $a - 1$ is
divisible by $4$). The minimal standard's period is $m - 1 = 2^{31} - 2
\approx 2\times10^9$; PCG64's is $2^{128}$. A simulation that consumes
more numbers than the period is recycling its own noise.

**Part c)** The corpse. Implement RANDU ($a = 65539$, $c = 0$,
$m = 2^{31}$) from seed $1$ and verify the *exact* integer identity
{eq}`eq-mc-randu`: $(x_{k+2} - 6x_{k+1} + 9x_k) \bmod 2^{31} = 0$ for
every $k$ over $10^4$ consecutive states. This is why RANDU's triples fall
on fifteen planes: the identity confines them {cite}`marsaglia1968`. Then
*see* it: plot 4000 consecutive triples $(u_k, u_{k+1}, u_{k+2})$ viewed
edge-on (matplotlib's 3-D axes at azimuth $-115°$, elevation $12°$, where
the planes align), beside the same picture for PCG64 triples, and animate a
slow rotation of the RANDU cloud: from almost every angle it looks like
noise, and from the right one it is fifteen sheets of paper.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    pm_first_three,
    [16807, 282475249, 1622650073],
    "the minimal standard's first three states, exact to the integer",
    rtol=0.0,
    atol=0.0,
)
validate.check(
    pm_ten_thousandth == 1043618065,
    "Park & Miller's own certification: state 10,000 from seed 1 is "
    "1043618065, bit-for-bit",
    f"got {pm_ten_thousandth}",
)
validate.check(
    toy_period == 32 and toy_distinct == 32,
    "the Hull–Dobell toy generator achieves its full period, visiting every "
    "residue mod 32 exactly once",
    f"period {toy_period}, distinct {toy_distinct}",
)
validate.check(
    randu_identity_max == 0,
    "RANDU's fifteen planes are an exact theorem: x_{k+2} = 6x_{k+1} − 9x_k "
    "(mod 2³¹) with zero residual over 10⁴ states",
    f"max residual {randu_identity_max}",
)

## Exercise 3 — Testing randomness like a skeptic

"Behaves random" is a statistical claim, and statistical claims are
tested. For genuinely uniform deviates on $[0, 1)$ the moments are
$\langle u^k\rangle = 1/(k+1)$; a histogram of $B$ equal bins should carry
counts whose $\chi^2 = \sum_b (n_b - N/B)^2/(N/B)$ follows the
$\chi^2$-distribution with $B - 1$ degrees of freedom; and consecutive
pairs should be uncorrelated, $\langle u_k u_{k+1}\rangle - 1/4 \to 0$ with
standard error of order $1/\sqrt N$... each test probes a different way a
generator can fail while passing the others (RANDU passes all three and
dies only in three dimensions, which is the lesson: tests rule out, never
certify).

**Part a)** Draw $N = 10^6$ PCG64 deviates (`numpy.random.default_rng(23)`)
and verify the first four moments $\langle u^k\rangle$, $k = 1,\dots,4$,
each within $4\sigma_k/\sqrt N$ of $1/(k+1)$, where $\sigma_k^2 =
1/(2k+1) - 1/(k+1)^2$ is the exact variance of $u^k$.

**Part b)** Bin the same sample into $B = 50$ equal bins with
`numpy.histogram` and verify $\chi^2$ lies inside the central 99.8% window
of $\chi^2_{49}$, $[\,$`scipy.stats.chi2.ppf(0.001, 49)`,
`scipy.stats.chi2.ppf(0.999, 49)`$\,]$. Verify the lag-1 correlation
$\langle u_k u_{k+1}\rangle - 1/4$ is within $4\cdot(1/4)/\sqrt N$ of zero
(the pair variance for uniforms is $\mathrm{Var}(u_ku_{k+1}) = 1/9 - 1/16
= 7/144 \approx (0.22)^2$; the $1/4$ window is comfortably wider).

**Part c)** Feed the same $\chi^2$ test (the `numpy.histogram` binning and
$\chi^2$ sum of Part b, unchanged) $10^6$ deviates from the toy
32-period generator of Exercise 2 (its $10^6$ samples are the same 32
values recycled 31,250 times). Verify its $\chi^2$ exceeds the PCG64 value
by a factor above $100$: a generator that fails, fails loudly, when the
test matches the failure.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    all(abs(p) < 4.0 for p in moment_pulls),
    "the first four moments of 10⁶ PCG64 deviates sit within 4σ of "
    "1/(k+1), each against its exact variance",
    f"pulls {[f'{p:+.2f}' for p in moment_pulls]}",
)
validate.check(
    chi2_lo < chi2_pcg < chi2_hi,
    "the 50-bin χ² lands inside the central 99.8% window of χ²₄₉: flat, but "
    "not suspiciously flat",
    f"χ² = {chi2_pcg:.1f} in [{chi2_lo:.1f}, {chi2_hi:.1f}]",
)
validate.check(
    abs(lag1) < 4.0 * 0.25 / np.sqrt(10**6),
    "consecutive deviates are uncorrelated at the 1/√N level",
    f"⟨u_k u_(k+1)⟩ − 1/4 = {lag1:+.2e}",
)
validate.check(
    chi2_toy > 100.0 * chi2_pcg,
    "the 32-period toy generator fails the same χ² test by more than two "
    "orders of magnitude: recycling is visible",
    f"ratio {chi2_toy / chi2_pcg:.0f}×",
)

## Exercise 4 — Manufacturing distributions

Physics rarely wants uniform numbers; it wants Boltzmann factors, decay
times, Gaussian noise. This exercise builds the three standard
transformations of the theory section, each validated against facts the
construction did not assume.

**Part a)** Inverse transform, {eq}`eq-mc-inverse`: from $N = 10^5$ uniform
deviates (`numpy.random.default_rng(31)`), form $X = -\ln(1 - U)$
(exponential, $\lambda = 1$: the distribution of radioactive decay times
with unit mean life). Verify the sample against the exponential
distribution with the Kolmogorov–Smirnov test,
`scipy.stats.kstest(x, "expon")`, demanding $p > 0.01$, and verify the
sample mean sits within $4/\sqrt N$ of $1$ (mean and standard deviation of
the unit exponential are both $1$).

**Part b)** Box–Muller, {eq}`eq-mc-boxmuller`: from two arrays of $5\times
10^4$ uniforms, build both Gaussian channels $Z_1, Z_2$, concatenate, and
verify with `scipy.stats.kstest(z, "norm")` at $p > 0.01$; verify the
sample variance within $4\sqrt{2/N}$ of $1$ (the variance of a sample
variance of $N$ Gaussians is $2/N$).

**Part c)** Rejection sampling for a density with no closed-form inverse:
the Wigner semicircle $p(x) = \tfrac{2}{\pi}\sqrt{1 - x^2}$ on $[-1, 1]$
(the eigenvalue density of the random matrices of
[§0.5](eigenvalues-svd.ipynb), one volume early). Propose $(x, y)$
uniformly in $[-1, 1]\times[0, 2/\pi]$, accept when $y < p(x)$, and verify
(i) the acceptance rate equals the area ratio $\frac{\pi/2\cdot(2/\pi)^2}
{2\cdot 2/\pi} = \pi/4$ within $4$ binomial standard errors, and (ii) the
accepted sample's variance meets the analytic
$\int x^2 p\,dx = 1/4$ within $4\sigma$ of the sample-variance error.
{numref}`fig-mc-transforms` shows all three histograms on their target
densities.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    ks_exp.pvalue > 0.01 and abs(exp_mean - 1.0) < 4.0 / np.sqrt(10**5),
    "the inverse transform delivers the unit exponential: KS accepts and the "
    "mean is 1 within 4σ",
    f"p = {ks_exp.pvalue:.3f}, mean = {exp_mean:.4f}",
)
validate.check(
    ks_norm.pvalue > 0.01 and abs(z_var - 1.0) < 4.0 * np.sqrt(2.0 / 10**5),
    "Box–Muller delivers exact standard Gaussians: KS accepts and the "
    "variance is 1 within 4σ",
    f"p = {ks_norm.pvalue:.3f}, var = {z_var:.4f}",
)
validate.close(
    accept_rate,
    np.pi / 4.0,
    "the rejection acceptance rate equals the area ratio π/4",
    atol=4.0 * accept_sem,
    rtol=0.0,
)
validate.close(
    semi_var,
    0.25,
    "the semicircle sample's variance meets the analytic ∫x²p dx = 1/4",
    rtol=2e-2,
)

## Exercise 5 — The 1/√N law, measured

Now the estimator itself. We integrate a completely explicit target,
$\int_0^1 e^x\,dx = e - 1$, with the mean-value estimator
{eq}`eq-mc-estimator`, and measure how its error falls with $N$: the claim
is a power law $N^{-1/2}$, so on log–log axes the RMS error over repeated
runs is a line of slope $-1/2$. Against it we run the trapezoid rule of
[§0.3](quadrature-differentiation.ipynb) on the same integral, whose error
falls as $N^{-2}$: in one dimension quadrature wins without contest, and
Exercise 6 shows why that verdict reverses in ten.

**Part a)** Implement the mean-value estimator. **Write this one
yourself** — the implementation is the lesson. (The Setup's
`mc_mean_value` is the reference; the two lines that matter are the mean
and the $(b-a)\,\mathrm{std}/\sqrt N$ error bar.) At $N = 10^4$
(`numpy.random.default_rng(47)`), verify the estimate lies within $4$ of
its *own* reported standard errors of $e - 1$: the estimator carries its
uncertainty with it, and the uncertainty is honest.

**Part b)** For $N \in \{10^2, 10^3, 10^4, 10^5, 10^6\}$, run $24$
independent repeats each (seeds $100\cdot j + i$ so every run is distinct
and reproducible), form the RMS error against the exact $e - 1$, and fit
$\log_{10}(\mathrm{RMS})$ vs $\log_{10} N$ with `numpy.polyfit` (degree 1,
the fitting machinery of [§0.8](fitting-least-squares.ipynb)). Verify the
slope is $-1/2$ within $\pm 0.08$.

**Part c)** Evaluate the trapezoid rule (`numpy.trapezoid` on a uniform
grid) at the same five $N$ and fit its slope: $-2$ within $\pm 0.1$.
{numref}`fig-mc-convergence` overlays both laws; the crossing favors the
grid everywhere in one dimension. Keep the figure in mind: Exercise 6
moves the same contest to $d$ dimensions, where the grid's exponent decays
as $-2/d$ and Monte Carlo's does not move.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(pull_single) < 4.0,
    "the estimator's own error bar is honest: the estimate sits within 4 of "
    "its reported standard errors of e − 1",
    f"pull {pull_single:+.2f}σ",
)
validate.close(
    slope_mc,
    -0.5,
    "the Monte Carlo error falls as N^(-1/2): the fitted log-log slope over "
    "five decades",
    atol=0.08,
    rtol=0.0,
)
validate.close(
    slope_trap,
    -2.0,
    "the trapezoid error falls as N^(-2) on the same integral: the "
    "one-dimensional benchmark to beat",
    atol=0.1,
    rtol=0.0,
)

## Exercise 6 — Where grids die: volumes in ten dimensions

The $d$-dimensional unit ball has exact volume

```{math}
:label: eq-mc-ball
V_d \;=\; \frac{\pi^{d/2}}{\Gamma(d/2 + 1)} ,
```

a formula with a strange story to tell: against the enclosing hypercube
$[-1,1]^d$ of volume $2^d$, the ball's share $V_d/2^d$ *collapses* with
dimension (from $\pi/4 \approx 0.785$ at $d = 2$ to $2.5\times10^{-3}$ at
$d = 10$): high-dimensional volume concentrates in corners. A product
trapezoid grid with even $10$ points per axis would need $10^{10}$
evaluations at $d = 10$; the dart estimator of Exercise 1, generalized
verbatim, needs only enough darts for the target precision, because its
$1/\sqrt N$ error never asks what $d$ is.

**Part a)** For $d \in \{2, 4, 6, 8, 10\}$, throw $N = 4\times10^5$ points
uniformly in $[-1, 1]^d$ (`numpy.random.default_rng(50 + d)`, shape
$(N, d)$), count the fraction $\hat p$ with $\sum_i x_i^2 \le 1$
(`numpy.mean` on the row-wise comparison of `numpy.sum(x**2, axis=1)`),
and estimate $V_d = 2^d\,\hat p$ with standard error
$2^d\sqrt{\hat p(1-\hat p)/N}$.

**Part b)** Verify every estimate against {eq}`eq-mc-ball` (via
`scipy.special.gamma`) within $4$ standard errors, and verify the
collapse: $\hat p(d)$ decreases monotonically and $\hat p(10) < 3\times
10^{-3}$. {numref}`fig-mc-hypersphere` shows the estimates riding the
exact curve.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    all(abs(p) < 4.0 for p in pulls_ball),
    "all five unit-ball volumes, d = 2 through 10, land within 4σ of "
    "π^(d/2)/Γ(d/2+1)",
    f"pulls {[f'{p:+.2f}' for p in pulls_ball]}",
)
validate.check(
    all(a > b for a, b in zip(p_hats, p_hats[1:])) and p_hats[-1] < 3e-3,
    "the corner collapse: the ball's share of the hypercube falls "
    "monotonically, to 0.25% by d = 10",
    f"p̂(10) = {p_hats[-1]:.2e}",
)

## Exercise 7 — Importance sampling, and the doorway to Metropolis

The mean-value estimator wastes samples wherever the integrand is small.
Equation {eq}`eq-mc-importance` fixes that by sampling where the integrand
lives. Our target is the completely explicit

$$I \;=\; \int_0^\infty x^{3/2}\,e^{-x}\,dx \;=\; \Gamma(5/2)
\;=\; \tfrac{3}{4}\sqrt{\pi} \;\approx\; 1.32934 ,$$

an integrand that is exponentially negligible beyond $x \approx 10$ yet
formally infinite in extent.

**Part a)** The naive route: truncate at $L = 25$ (where the integrand is
below $10^{-8}$) and apply the mean-value estimator on $[0, L]$ with
$N = 10^5$ uniform samples (`numpy.random.default_rng(61)`). It works, and
wastefully: most samples land where $e^{-x}$ has already killed the
integrand. Record the estimate and its standard error.

**Part b)** The importance route, with a weight *shaped like* the
integrand: the Gamma$(2,1)$ density $w(x) = x\,e^{-x}$, which one samples
with the tools already built, as the sum of two independent unit
exponentials, $X = -\ln(1-U_1) - \ln(1-U_2)$ (the inverse transform of
Exercise 4, twice; seed $62$, same $N$). The estimator averages
$f(X)/w(X) = \sqrt X$, whose variance under $w$ is $\mathbb E[X] - I^2 =
2 - \Gamma(5/2)^2 \approx 0.233$, against the naive route's
$\approx 7.6$ (after the $L^2$ factor). Verify both estimates agree with
$\Gamma(5/2)$ within $4$ of their own standard errors, and verify the
variance reduction: the naive standard error exceeds the matched-weight
one by a factor above $5$. The weight did not change the answer (it
cannot: {eq}`eq-mc-importance` is an identity); it changed the *cost* of
the answer. A first, tempting choice of weight, plain $w = e^{-x}$, buys
almost nothing here (a factor $1.3$: its ratio $f/w = x^{3/2}$ is
unbounded, so its variance stays large) — this notebook's own drafting
made exactly that mistake, and the variance formula caught it. Matching
the weight to the integrand's *shape*, not merely its tail, is the craft.

**Part c)** Push the logic to its end: the *perfect* weight is the
normalized integrand itself, $w^\star(x) = x^{3/2}e^{-x}/\Gamma(5/2)$ (the
Gamma$(5/2, 1)$ density, drawn with `numpy.random.Generator.gamma(2.5)`,
seed $63$). Then $f/w^\star = \Gamma(5/2)$ is *constant*: every sample
returns the exact answer and the estimator's spread collapses to rounding
(verify the sample standard deviation of the ratios is below $10^{-12}$).
The catch is the point: building $w^\star$ required already knowing the
normalization $\Gamma(5/2)$, which *is* the integral. Benchmark everything
against deterministic quadrature, `scipy.integrate.quad` of
$x^{3/2}e^{-x}$ on $(0, \infty)$, which agrees with $\Gamma(5/2)$ to
thirteen digits: in one dimension, quadrature still reigns. The
limitation to carry forward is Part c)'s circle: importance sampling
required a weight we could *sample directly*. The Boltzmann weight
$e^{-\beta E(\mathbf s)}/Z$ of statistical mechanics is exactly the weight
one wants and exactly the one no inverse transform reaches ($Z$ itself is
the unknown). The machine that samples it anyway, one accepted move at a
time, is the Metropolis algorithm, built in
[§5.8](../05-classical-stat-mech/partition-function.ipynb) where its
physics lives, and refined ever after (checkerboard sweeps in
[§5.10](../05-classical-stat-mech/ising-emergence-universality.ipynb),
staging for ring polymers in
[§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb)).
This notebook ends at that doorway on purpose.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(est_naive - GAMMA_52) < 4.0 * sem_naive
    and abs(est_imp - GAMMA_52) < 4.0 * sem_imp,
    "both estimators agree with Γ(5/2) within 4 of their own standard "
    "errors: the weight changes cost, never the answer",
    f"naive pull {(est_naive - GAMMA_52) / sem_naive:+.2f}σ, "
    f"importance pull {(est_imp - GAMMA_52) / sem_imp:+.2f}σ",
)
validate.check(
    sem_naive / sem_imp > 5.0,
    "the shape-matched Gamma(2,1) weight cuts the standard error more than "
    "5× at identical N",
    f"factor {sem_naive / sem_imp:.1f}×",
)
validate.check(
    star_spread < 1e-12,
    "the perfect weight w* ∝ f has ZERO variance: every sample returns "
    "Γ(5/2) exactly — and building it required knowing the answer",
    f"ratio spread {star_spread:.1e}",
)
validate.close(
    quad_val,
    GAMMA_52,
    "scipy.integrate.quad confirms Γ(5/2) to thirteen digits: in one "
    "dimension, deterministic quadrature still reigns",
    rtol=1e-12,
)

## Notebook summary

- A million uniform darts estimated $\pi$ within four binomial standard
  errors, and quadrupling precision cost a hundredfold work: the
  $1/\sqrt N$ exchange rate, met before its formalism.
- The Park–Miller minimal standard {eq}`eq-mc-lcg`, in exact integer
  arithmetic, reproduced its designers' certification bit-for-bit
  (states $16807, 282475249, 1622650073$; state $10{,}000 = 1043618065$);
  the Hull–Dobell toy generator achieved its full period of 32; and
  RANDU's fifteen planes were verified as the *exact* theorem
  {eq}`eq-mc-randu`, with zero integer residual over $10^4$ states, then
  seen edge-on and in rotation.
- Randomness testing ruled like a skeptic: four moments within $4\sigma$,
  the 50-bin $\chi^2$ inside the central 99.8% window of $\chi^2_{49}$,
  lag-1 correlation at the $1/\sqrt N$ level, and the 32-period generator
  failing the same $\chi^2$ by more than two orders of magnitude.
- Uniform deviates became exponential ({eq}`eq-mc-inverse`, KS-accepted,
  mean $1$), Gaussian ({eq}`eq-mc-boxmuller`, KS-accepted, variance $1$),
  and semicircular (rejection; acceptance rate $\pi/4$, variance $1/4$).
- The mean-value estimator {eq}`eq-mc-estimator` carried an honest error
  bar and a measured slope of $-1/2$ over five decades of $N$, against the
  trapezoid rule's $-2$: the one-dimensional verdict, reversed in high
  dimension by the unit-ball volumes of {eq}`eq-mc-ball`, all five within
  $4\sigma$ up to $d = 10$ where the grid could not have been afforded.
- Importance sampling {eq}`eq-mc-importance` with the shape-matched
  Gamma$(2,1)$ weight cut the standard error of $\Gamma(5/2)$ more than
  fivefold at identical cost (a merely tail-matched $e^{-x}$ bought a
  factor $1.3$: shape is the craft); the perfect weight collapsed the
  variance to rounding at the price of already knowing the answer; and
  that circle named the machine this notebook
  deliberately stops before: Metropolis, built in
  [§5.8](../05-classical-stat-mech/partition-function.ipynb).

## Outlook

- **The central limit theorem as physics.** The $1/\sqrt N$ measured here
  is derived in [§5.3](../05-classical-stat-mech/large-n-limit.ipynb) and
  becomes the reason thermodynamic quantities are sharp;
  [§5.2](../05-classical-stat-mech/probability.ipynb) supplies the
  probability language this notebook used informally.
- **Markov-chain Monte Carlo.** When the weight cannot be sampled
  directly, Metropolis
  ([§5.8](../05-classical-stat-mech/partition-function.ipynb)) samples it
  through a chain of accepted moves, at the price of correlated samples;
  the autocorrelation bookkeeping that prices that correlation honestly is
  built in
  [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb).
- **Quasi-random sequences.** Low-discrepancy (Sobol) points beat
  $1/\sqrt N$ for smooth integrands in moderate dimension
  (`scipy.stats.qmc`); the price is the loss of the honest statistical
  error bar, and the trade is worth knowing by name.
- **Generators as a research subject.** PCG64 is not the last word;
  counter-based and cryptographic generators matter where streams must be
  split across parallel workers without correlation. Numerical Recipes
  ch. 7 {cite}`numrecipes` is the course's standing reference.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()